In [9]:
# Импортируем библиотеки
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch import nn, optim
import numpy as np
import random
import os, json, pandas as pd
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

# Настройка повторяемости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Seed: {SEED}")

# Определяем устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

torch: 2.9.0+cpu
torchvision: 0.24.0+cpu
Seed: 42
Устройство: cpu


In [10]:
# Трансформация
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Загрузка датасета EMNIST (split='balanced')
train_dataset_full = torchvision.datasets.EMNIST(
    root="./data",
    split="balanced",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.EMNIST(
    root="./data",
    split="balanced",
    train=False,
    download=True,
    transform=transform
)

# Разбиение 80/20
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

# Воспроизводимое разбиение
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

# DataLoader (размер батча)
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Вывод информации о датасете
print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")
print(f"Количество классов: {len(train_dataset_full.classes)}")

# sanity-check 
x_batch, y_batch = next(iter(train_loader))

print("Batch size:", x_batch.size(0))
print("x.shape:", x_batch.shape)  
print("y.shape:", y_batch.shape)  
print("Тип x:", x_batch.dtype)
print("Тип y:", y_batch.dtype)
print("Min x:", x_batch.min().item())
print("Max x:", x_batch.max().item())
print("Уникальные классы в батче:", torch.unique(y_batch))

100.0%


Train: 90240
Val: 22560
Test: 18800
Количество классов: 47
Batch size: 128
x.shape: torch.Size([128, 1, 28, 28])
y.shape: torch.Size([128])
Тип x: torch.float32
Тип y: torch.int64
Min x: -1.0
Max x: 1.0
Уникальные классы в батче: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 17, 18,
        19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
        38, 39, 40, 41, 42, 43, 44, 45, 46])


In [11]:
# Модель MLP (Flatten, Linear, …, logits).
class MLP(nn.Module):
    def __init__(self,
                 input_dim=28*28,
                 hidden_dims=[256],
                 num_classes=47,
                 dropout=0.0,
                 batchnorm=False):
        super().__init__()

        layers = [nn.Flatten()]
        prev_dim = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))

            if batchnorm:
                layers.append(nn.BatchNorm1d(h))

            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            prev_dim = h

        layers.append(nn.Linear(prev_dim, num_classes))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)
    
# Инициализация модели, функции потерь и оптимизатора
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Функция Accuracy
def accuracy_fn(logits, targets):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == targets).sum().item()
    return correct / targets.size(0)

In [12]:
# Функция обучения одной эпохи
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += x.size(0)

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc

In [13]:
# Функция оценки
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)

            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == y).sum().item()
            total_samples += x.size(0)

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc

In [14]:
num_epochs = 10

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

# Основной цикл обучения с логированием
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )

Epoch [1/10] | Train Loss: 1.2899, Train Acc: 0.6367 | Val Loss: 0.9221, Val Acc: 0.7281
Epoch [2/10] | Train Loss: 0.7739, Train Acc: 0.7662 | Val Loss: 0.7356, Val Acc: 0.7731
Epoch [3/10] | Train Loss: 0.6360, Train Acc: 0.7995 | Val Loss: 0.6434, Val Acc: 0.8018
Epoch [4/10] | Train Loss: 0.5668, Train Acc: 0.8171 | Val Loss: 0.6005, Val Acc: 0.8112
Epoch [5/10] | Train Loss: 0.5230, Train Acc: 0.8280 | Val Loss: 0.5773, Val Acc: 0.8148
Epoch [6/10] | Train Loss: 0.4907, Train Acc: 0.8370 | Val Loss: 0.5620, Val Acc: 0.8190
Epoch [7/10] | Train Loss: 0.4635, Train Acc: 0.8437 | Val Loss: 0.5430, Val Acc: 0.8263
Epoch [8/10] | Train Loss: 0.4448, Train Acc: 0.8489 | Val Loss: 0.5575, Val Acc: 0.8226
Epoch [9/10] | Train Loss: 0.4280, Train Acc: 0.8529 | Val Loss: 0.5667, Val Acc: 0.8214
Epoch [10/10] | Train Loss: 0.4134, Train Acc: 0.8563 | Val Loss: 0.5535, Val Acc: 0.8217


In [15]:
def run_experiment(exp_id,
                   hidden_dims,
                   dropout,
                   batchnorm,
                   num_epochs=15,
                   early_stopping=False,
                   patience=4):

    model = MLP(
        hidden_dims=hidden_dims,
        dropout=dropout,
        batchnorm=batchnorm
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = {"train_loss": [], "val_loss": [],
               "train_acc": [], "val_acc": []}

    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(num_epochs):

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping and patience_counter >= patience:
            break

    result = {
        "experiment_id": exp_id,
        "model": model,
        "best_state": best_state,
        "history": history,
        "epochs_trained": epoch + 1,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "dropout": dropout,
        "batchnorm": batchnorm,
        "hidden_dims": hidden_dims
    }

    return result

In [ ]:
results = []

# E1 base
results.append(run_experiment(
    exp_id="E1",
    hidden_dims=[512, 256],
    dropout=0.0,
    batchnorm=False
))

# E2 Dropout
results.append(run_experiment(
    exp_id="E2",
    hidden_dims=[512, 256],
    dropout=0.3,
    batchnorm=False
))

# E3 BatchNorm
results.append(run_experiment(
    exp_id="E3",
    hidden_dims=[512, 256],
    dropout=0.0,
    batchnorm=True
))

In [17]:
e2 = next(r for r in results if r["experiment_id"] == "E2")
e3 = next(r for r in results if r["experiment_id"] == "E3")

best_regularized = e2 if e2["best_val_accuracy"] > e3["best_val_accuracy"] else e3

results.append(run_experiment(
    exp_id="E4",
    hidden_dims=best_regularized["hidden_dims"],
    dropout=best_regularized["dropout"],
    batchnorm=best_regularized["batchnorm"],
    early_stopping=True,
    patience=4
))

In [18]:
best_exp = max(results, key=lambda r: r["best_val_accuracy"])

In [19]:
best_model = MLP(
    hidden_dims=best_exp["hidden_dims"],
    dropout=best_exp["dropout"],
    batchnorm=best_exp["batchnorm"]
).to(device)

best_model.load_state_dict(best_exp["best_state"])

test_loss, test_acc = evaluate(
    best_model, test_loader,
    nn.CrossEntropyLoss(), device
)

print("Best experiment:", best_exp["experiment_id"])
print("Test accuracy:", test_acc)

Best experiment: E4
Test accuracy: 0.8456382978723405


In [ ]:
# ЧАСТЬ B (S09)

# Функция запуска эксперимента S09
def run_optimization_experiment(exp_id, 
                                 hidden_dims, 
                                 dropout, 
                                 batchnorm,
                                 optimizer_name,
                                 lr,
                                 momentum=0.0,
                                 weight_decay=0.0,
                                 num_epochs=8):
    
    model = MLP(
        hidden_dims=hidden_dims,
        dropout=dropout,
        batchnorm=batchnorm
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    
    # Настройка оптимизатора
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(), 
            lr=lr, 
            weight_decay=weight_decay
        )
    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(), 
            lr=lr, 
            momentum=momentum, 
            weight_decay=weight_decay
        )
    
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    best_val_acc = 0.0
    best_val_loss = float("inf")
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
    
    result = {
        "experiment_id": exp_id,
        "model": model,
        "history": history,
        "epochs_trained": num_epochs,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
        "optimizer": optimizer_name,
        "lr": lr,
        "momentum": momentum,
        "weight_decay": weight_decay,
        "hidden_dims": hidden_dims,
        "dropout": dropout,
        "batchnorm": batchnorm
    }
    
    return result

# Запуск O1, O2, O3

# Берём архитектуру из E4
e4 = next(r for r in results if r["experiment_id"] == "E4")

# O1: LR слишком большой (1e-1)
results.append(run_optimization_experiment(
    exp_id="O1",
    hidden_dims=e4["hidden_dims"],
    dropout=e4["dropout"],
    batchnorm=e4["batchnorm"],
    optimizer_name="Adam",
    lr=1e-1,
    num_epochs=8
))

# O2: LR слишком маленький (1e-5)
results.append(run_optimization_experiment(
    exp_id="O2",
    hidden_dims=e4["hidden_dims"],
    dropout=e4["dropout"],
    batchnorm=e4["batchnorm"],
    optimizer_name="Adam",
    lr=1e-5,
    num_epochs=8
))

# O3: SGD + momentum + weight_decay
results.append(run_optimization_experiment(
    exp_id="O3",
    hidden_dims=e4["hidden_dims"],
    dropout=e4["dropout"],
    batchnorm=e4["batchnorm"],
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
    num_epochs=15
))

In [21]:
base_path = "./artifacts"
fig_path = os.path.join(base_path, "figures")
os.makedirs(fig_path, exist_ok=True)

In [32]:
def build_model_summary(r):
    hidden = r["hidden_dims"]

    if len(hidden) == 1:
        arch_text = f"Однослойный MLP: 1 скрытый слой ({hidden[0]} нейронов)"
    else:
        arch_text = (
            f"MLP с {len(hidden)} скрытыми слоями "
            f"({', '.join(map(str, hidden))} нейронов)"
        )

    if r["batchnorm"]:
        reg_text = "с BatchNorm"
    elif r["dropout"] > 0:
        reg_text = f"с Dropout={r['dropout']}"
    else:
        reg_text = "без регуляризации"

    if r["experiment_id"] == "E4":
        reg_text += " и EarlyStopping"

    return f"{arch_text}, ReLU, {reg_text}"


rows = []

for r in results:
    rows.append({
        "experiment_id": r["experiment_id"],
        "dataset": "EMNIST",
        "seed": SEED,
        "model_summary": build_model_summary(r),
        "optimizer": r.get("optimizer", "Adam"),  # по умолчанию Adam для E1-E4
        "lr": r.get("lr", 1e-3),
        "momentum": r.get("momentum", 0.0),
        "weight_decay": r.get("weight_decay", 0.0),
        "dropout": r["dropout"],
        "batchnorm": r["batchnorm"],
        "epochs_trained": r["epochs_trained"],
        "best_val_accuracy": r["best_val_accuracy"],
        "best_val_loss": r["best_val_loss"]
    })

df = pd.DataFrame(rows)
df.to_csv(os.path.join(base_path, "runs.csv"), index=False, encoding="utf-8-sig")

In [33]:
torch.save(best_exp["best_state"],
           os.path.join(base_path, "best_model.pt"))

In [34]:
best_config = {
    "dataset": "EMNIST",
    "seed": SEED,
    "hidden_dims": best_exp["hidden_dims"],
    "dropout": best_exp["dropout"],
    "batchnorm": best_exp["batchnorm"]
}

with open(os.path.join(base_path, "best_config.json"), "w") as f:
    json.dump(best_config, f, indent=4)

In [35]:
hist = best_exp["history"]

plt.figure()
plt.plot(hist["train_loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.legend()
plt.title("Best model loss curves")
plt.savefig(os.path.join(fig_path, "curves_best.png"))
plt.close()

In [ ]:
# График LR extremes
o1 = next(r for r in results if r["experiment_id"] == "O1")
o2 = next(r for r in results if r["experiment_id"] == "O2")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# O1
axes[0].plot(o1["history"]["train_loss"], label="train_loss", color="red")
axes[0].plot(o1["history"]["val_loss"], label="val_loss", color="orange")
axes[0].set_title(f"O1: LR слишком большой ({o1['lr']})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# O2
axes[1].plot(o2["history"]["train_loss"], label="train_loss", color="blue")
axes[1].plot(o2["history"]["val_loss"], label="val_loss", color="green")
axes[1].set_title(f"O2: LR слишком маленький ({o2['lr']})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(fig_path, "curves_lr_extremes.png"), dpi=150)
plt.close()

print("График curves_lr_extremes.png сохранён")

График curves_lr_extremes.png сохранён
